In [ ]:
%load_ext autoreload
%autoreload 2

# Palisades Wildfires

This notebook demonstrates how to download and preprocess Sentinel-1 and Sentinel-2 imagery, and run building damage assessment using pre-trained models from the Hugging Face Hub.

To identify relevant acquisitions, we recommend using the [Copernicus Browser](https://browser.dataspace.copernicus.eu/) or [Google Earth Engine (GEE)](https://earthengine.google.com/) to inspect available scenes before downloading.

> **Prerequisites**
> - A valid **GEE account** for Sentinel-1 download
> - A valid **ESA Copernicus account** for Sentinel-2 download via [phidown](https://esa-philab.github.io/phidown/)

### Notebook Structure

1. **Define Area of Interest** — Specify the geographic extent of the area affected by the Palisades Wildfires.
2. **Download Data**
   - 2.1 Download Sentinel-1 data via GEE
   - 2.2 Download Sentinel-2 data via phidown
3. **Preprocess Data** — Reproject both sensors onto a shared grid at 4 m resolution.
4. **Run Inference** — Run building damage assessment using the default models from the Hugging Face Hub.

## Setup

In [ ]:
import geopandas as gpd
from rasterio.enums import Resampling
import rioxarray as rxr
from shapely import box

from src.constants import DATA_PATH
from src.data.sentinel.sentinel1 import download_sentinel1
from src.data.sentinel.sentinel2 import download_sentinel2
from src.data.sentinel.utils import save_raster
from src.utils.gee import init_gee
from src.utils.geometry import get_best_utm_crs, reproject_geo

# Initialize Google Earth Engine
PROJECT_GEE = ...  # Add your GEE project
init_gee(project=PROJECT_GEE)

# Define folder to store the data for the given experiment
folder_exp = DATA_PATH / 'inference' / 'palisades_wildfires'
folder_exp.mkdir(exist_ok=True, parents=True)

## Area of interest

In [ ]:
gdf = gpd.read_file(folder_exp / 'fires_extent.geojson') # TODO: check where this file comes from
geo = box(*gdf.iloc[0].geometry.bounds)

## Download Sentinel-1

In [ ]:
S1_PRE_DATE = "2024-11-27"
S1_POST_DATE = "2025-02-07"
S1_ORBIT_NUMBER = 137

download_sentinel1(folder_exp, date=S1_PRE_DATE, orbit_number=S1_ORBIT_NUMBER, geo=geo, bands=['VV', 'VH'])
download_sentinel1(folder_exp, date=S1_POST_DATE, orbit_number=S1_ORBIT_NUMBER, geo=geo, bands=['VV', 'VH'])

## Download Sentinel-2

In [ ]:
S2_PRE_DATE = "2024-11-28"
S2_POST_DATE = "2025-02-01"
S2_MGRS = "11SLT"
geo_utm = reproject_geo(geo, 'EPSG:4326', get_best_utm_crs(geo)) # Important to reproject the geometry to same CRS as

download_sentinel2(folder_exp, date=S2_PRE_DATE, mgrs=S2_MGRS, geo=geo_utm)
download_sentinel2(folder_exp, date=S2_POST_DATE, mgrs=S2_MGRS, geo=geo_utm)

## Preprocess Data
Reproject all data on a shared grid at 4m resolution

In [ ]:
fp_s2_pre = folder_exp / f's2_{S2_PRE_DATE}_{S2_MGRS}_l2a.tif'
fp_s2_post= folder_exp / f's2_{S2_POST_DATE}_{S2_MGRS}_l2a.tif'
fp_s1_pre = folder_exp / f's1_{S1_PRE_DATE}_o{S1_ORBIT_NUMBER}.tif'
fp_s1_post = folder_exp / f's1_{S1_POST_DATE}_o{S1_ORBIT_NUMBER}.tif'



# Reproject S2 pre to 4m resolution
xa = rxr.open_rasterio(fp_s2_pre)
target_fp = fp_s2_pre.with_name(fp_s2_pre.stem + '_4m.tif')
if target_fp.exists():
    print(f"File {target_fp} already exists, skipping reprojection.")
else:
    xa_proj = xa.rio.reproject(xa.rio.crs, resolution=4.0, resampling=Resampling.lanczos)
    save_raster(xa_proj, target_fp)

# Reproject all other rasters to match exactly the same grid
target = rxr.open_rasterio(target_fp, chunks=True)
for fp in [fp_s2_post, fp_s1_pre, fp_s1_post]:

    new_fp = fp.with_name(fp.stem + '_4m.tif')
    if new_fp.exists():
        print(f"File {new_fp} already exists, skipping reprojection.")
        continue
    xa = rxr.open_rasterio(fp, chunks=True)
    xa_proj = xa.rio.reproject_match(target, resampling=Resampling.lanczos)
    xa_proj = xa_proj.fillna(0)  # fill no data with 0 (for rare case of S1 that has holes due to mosaicking)
    xa_proj = xa_proj.assign_coords({'x': target.x, 'y': target.y})
    save_raster(xa_proj, new_fp)

## Run Inference
From pretrained models uploaded on Hugging Face Hub.

In [ ]:
from src.inference.from_hub import InferenceFromHub

fp_s2_pre_ready = fp_s2_pre.with_name(fp_s2_pre.stem + '_4m.tif')
fp_s2_post_ready = fp_s2_post.with_name(fp_s2_post.stem + '_4m.tif')
fp_s1_pre_ready = fp_s1_pre.with_name(fp_s1_pre.stem + '_4m.tif')
fp_s1_post_ready = fp_s1_post.with_name(fp_s1_post.stem + '_4m.tif')

# by default, use the 3 models with different seeds for both localization and damage from the HF repo
inferor = InferenceFromHub()

In [ ]:
output_fp = folder_exp / "predictions_palisades_fire.tif"
inferor.run_inference(fp_s2_pre_ready, fp_s2_post_ready, fp_s1_pre_ready, fp_s1_post_ready, output_fp)